# ENA24 — Faster R-CNN


## 1. Executive summary

The experimental reasoning retained is:

- **Baseline**: initial training for 10 epochs.
- **Verification**: extend training to check whether validation continues to improve.
- **Observation**: `val_loss` reaches a plateau; extending training alone is not worthwhile.
- **Pivot**: test a **data augmentation** approach.
- **Result**: improvement compared with the baseline.
- **Final attempt**: fine-tuning.
- **Conclusion**: fine-tuning does not outperform data augmentation, so the best selected pipeline is the one with **data augmentation**.


## 2. Observed quantitative summary

| approach          |   observed_epochs |   observed_best_epoch |   observed_best_val_loss |   observed_last_val_loss | comment                                                           |
|:------------------|------------------:|----------------------:|-------------------------:|-------------------------:|:------------------------------------------------------------------|
| Baseline          |                10 |                     7 |                   0.0888 |                   0.0896 | Visible validation plateau from the last epochs onward.           |
| Data augmentation |                10 |                    10 |                   0.0860 |                   0.0860 | Best generalization among the documented runs.                    |
| Fine-tuning       |                 3 |                     3 |                   0.0864 |                   0.0864 | Slight local improvement, but not better than data augmentation.  |


In [ ]:
# Figure comparing the observed validation losses

In [ ]:
# Composite of the graphs visible in the outputs of the provided labs

## 3. Imports

In [ ]:
import os
import json
import time
import random
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 4. Global configuration

In [ ]:
# =========================
# General configuration
# =========================
SEED = 42
BATCH_SIZE = 4
VAL_RATIO = 0.2
NUM_WORKERS = 4

# Baseline / augmentation training
NUM_EPOCHS = 10
LR = 5e-3
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
STEP_SIZE = 5
GAMMA = 0.1

# Fine-tuning
FINE_TUNE_EPOCHS = 8
FINE_TUNE_LR = 1e-3
FINE_TUNE_MOMENTUM = 0.9
FINE_TUNE_WEIGHT_DECAY = 1e-4
PLATEAU_FACTOR = 0.5
PLATEAU_PATIENCE = 1
EARLY_STOPPING_PATIENCE = 3
MIN_DELTA = 1e-4

# Class filtering
FILTER_CATEGORY_IDS = {8, 9}

# Checkpoints
SYNC_EVERY_N_EPOCHS = 1

# Runtime
USE_AMP = torch.cuda.is_available()
pin_memory = torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print("Device:", device)
print("AMP:", USE_AMP)


## 5. Data paths and local copy

This cell keeps the spirit of the provided Colab notebooks: data on Drive, local copy before training to reduce I/O.


In [ ]:
# =========================
# Paths to adapt
# =========================
# Colab / Drive version
DRIVE_ROOT = Path("/content/drive/MyDrive/ena24")
DRIVE_IMAGES = DRIVE_ROOT / "images"
DRIVE_JSON = DRIVE_ROOT / "metadata.json"

LOCAL_ROOT = Path("/content/ena24")
LOCAL_IMAGES = LOCAL_ROOT / "images"
LOCAL_JSON = LOCAL_ROOT / "metadata.json"

# To run locally outside Colab, simply replace these paths
IMAGE_DIR = LOCAL_IMAGES
ANN_PATH = LOCAL_JSON

# Recommended local copy to avoid slow Drive I/O
def copy_data_locally(drive_images=DRIVE_IMAGES, drive_json=DRIVE_JSON,
                      local_images=LOCAL_IMAGES, local_json=LOCAL_JSON):
    local_images.parent.mkdir(parents=True, exist_ok=True)

    if not local_json.exists():
        shutil.copy2(drive_json, local_json)

    if not local_images.exists():
        shutil.copytree(drive_images, local_images)

    print("JSON exists:", local_json.exists())
    print("Images exist:", local_images.exists())

# Uncomment in Colab after mounting Drive
# from google.colab import drive
# drive.mount("/content/drive", force_remount=False)
# copy_data_locally()


## 6. ENA24 PyTorch dataset

The dataset follows the structure of the provided labs, with additional optional augmentation parameters for training.


In [ ]:
class ENA24DetectionDataset(Dataset):
    def __init__(self, image_dir, ann_path, filter_category_ids=None, training=False,
                 hflip_prob=0.0, brightness_jitter=0.0, contrast_jitter=0.0):
        self.image_dir = Path(image_dir)
        self.ann_path = Path(ann_path)
        self.filter_category_ids = set(filter_category_ids or [])
        self.training = training
        self.hflip_prob = float(hflip_prob)
        self.brightness_jitter = float(brightness_jitter)
        self.contrast_jitter = float(contrast_jitter)

        with open(self.ann_path, "r") as f:
            data = json.load(f)

        self.images = data["images"]
        self.annotations = data["annotations"]
        self.categories = data["categories"]

        self.image_id_to_info = {img["id"]: img for img in self.images}

        self.cat_id_to_name = {
            cat["id"]: cat["name"]
            for cat in self.categories
            if cat["id"] not in self.filter_category_ids
        }

        kept_cat_ids = sorted(self.cat_id_to_name.keys())
        self.cat_id_to_label = {cat_id: i + 1 for i, cat_id in enumerate(kept_cat_ids)}
        self.label_to_cat_id = {v: k for k, v in self.cat_id_to_label.items()}
        self.label_to_name = {
            label: self.cat_id_to_name[cat_id]
            for cat_id, label in self.cat_id_to_label.items()
        }

        anns_per_image = defaultdict(list)
        for ann in self.annotations:
            cat_id = ann["category_id"]
            if cat_id in self.filter_category_ids:
                continue
            if cat_id not in self.cat_id_to_label:
                continue
            anns_per_image[ann["image_id"]].append(ann)

        self.samples = []
        self.missing_files = 0
        self.empty_after_filter = 0

        for image_id, anns in anns_per_image.items():
            img_info = self.image_id_to_info.get(image_id)
            if img_info is None:
                continue

            img_path = self.image_dir / img_info["file_name"]
            if not img_path.is_file():
                self.missing_files += 1
                continue

            if len(anns) == 0:
                self.empty_after_filter += 1
                continue

            self.samples.append((image_id, img_path, anns))

        print(f"Retained images              : {len(self.samples)}")
        print(f"Retained classes             : {len(self.cat_id_to_label)}")
        print(f"Missing images               : {self.missing_files}")
        print(f"Empty images after filtering : {self.empty_after_filter}")

    def __len__(self):
        return len(self.samples)

    def _apply_augmentations(self, image, boxes):
        # Horizontal flip
        if self.training and self.hflip_prob > 0 and random.random() < self.hflip_prob:
            image = F.hflip(image)
            w, _ = image.size
            if len(boxes) > 0:
                x1 = boxes[:, 0].clone()
                x2 = boxes[:, 2].clone()
                boxes[:, 0] = w - x2
                boxes[:, 2] = w - x1

        # Simple photometric augmentation
        if self.training and self.brightness_jitter > 0:
            factor = 1.0 + random.uniform(-self.brightness_jitter, self.brightness_jitter)
            image = F.adjust_brightness(image, factor)

        if self.training and self.contrast_jitter > 0:
            factor = 1.0 + random.uniform(-self.contrast_jitter, self.contrast_jitter)
            image = F.adjust_contrast(image, factor)

        return image, boxes

    def __getitem__(self, idx):
        image_id, img_path, anns = self.samples[idx]

        with Image.open(img_path) as img:
            image = img.convert("RGB")

        w, h = image.size
        boxes, labels, areas, iscrowd = [], [], [], []

        for ann in anns:
            x, y, bw, bh = ann["bbox"]

            x1 = max(0.0, x)
            y1 = max(0.0, y)
            x2 = min(float(w), x + bw)
            y2 = min(float(h), y + bh)

            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(self.cat_id_to_label[ann["category_id"]])
            areas.append((x2 - x1) * (y2 - y1))
            iscrowd.append(0)

        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        image, boxes = self._apply_augmentations(image, boxes)

        image = F.pil_to_tensor(image)
        image = F.convert_image_dtype(image, dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([int(image_id)], dtype=torch.int64),
            "area": areas,
            "iscrowd": iscrowd,
        }
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

## 7. Train / validation split and DataLoaders

The split is fixed by `SEED` to keep a fair comparison between baseline, data augmentation, and fine-tuning.


In [ ]:
# =========================
# Build datasets / dataloaders
# =========================
# Baseline
baseline_dataset = ENA24DetectionDataset(
    image_dir=IMAGE_DIR,
    ann_path=ANN_PATH,
    filter_category_ids=FILTER_CATEGORY_IDS,
    training=False,
)

num_classes = len(baseline_dataset.cat_id_to_label) + 1
n_total = len(baseline_dataset)
n_val = max(1, int(n_total * VAL_RATIO))
n_train = n_total - n_val

train_indices, val_indices = random_split(
    range(n_total),
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

# Important:
# We rebuild 2 distinct datasets to keep the same split
# while applying augmentation only on the training set.
baseline_train_full = ENA24DetectionDataset(
    image_dir=IMAGE_DIR,
    ann_path=ANN_PATH,
    filter_category_ids=FILTER_CATEGORY_IDS,
    training=False,
)
baseline_val_full = ENA24DetectionDataset(
    image_dir=IMAGE_DIR,
    ann_path=ANN_PATH,
    filter_category_ids=FILTER_CATEGORY_IDS,
    training=False,
)

aug_train_full = ENA24DetectionDataset(
    image_dir=IMAGE_DIR,
    ann_path=ANN_PATH,
    filter_category_ids=FILTER_CATEGORY_IDS,
    training=True,
    hflip_prob=0.5,
    brightness_jitter=0.15,
    contrast_jitter=0.15,
)
aug_val_full = ENA24DetectionDataset(
    image_dir=IMAGE_DIR,
    ann_path=ANN_PATH,
    filter_category_ids=FILTER_CATEGORY_IDS,
    training=False,
)

from torch.utils.data import Subset
baseline_train_dataset = Subset(baseline_train_full, train_indices.indices)
baseline_val_dataset = Subset(baseline_val_full, val_indices.indices)
aug_train_dataset = Subset(aug_train_full, train_indices.indices)
aug_val_dataset = Subset(aug_val_full, val_indices.indices)

baseline_train_loader = DataLoader(
    baseline_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=pin_memory,
    persistent_workers=(NUM_WORKERS > 0),
)

baseline_val_loader = DataLoader(
    baseline_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=pin_memory,
    persistent_workers=(NUM_WORKERS > 0),
)

aug_train_loader = DataLoader(
    aug_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=pin_memory,
    persistent_workers=(NUM_WORKERS > 0),
)

aug_val_loader = DataLoader(
    aug_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=pin_memory,
    persistent_workers=(NUM_WORKERS > 0),
)

print("num_classes =", num_classes)
print("Train:", len(baseline_train_dataset))
print("Val  :", len(baseline_val_dataset))

## 8. Model, training functions, and checkpoints

In [ ]:
def get_faster_rcnn_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def move_batch_to_device(images, targets, device):
    images = [img.to(device, non_blocking=True) for img in images]
    targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
    return images, targets

def train_one_epoch(model, optimizer, data_loader, device, epoch, scaler=None, print_freq=20):
    model.train()
    running_loss = 0.0
    n_steps = 0

    for step, (images, targets) in enumerate(data_loader):
        images, targets = move_batch_to_device(images, targets, device)
        optimizer.zero_grad(set_to_none=True)

        if USE_AMP:
            with torch.amp.autocast(device_type="cuda"):
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
        else:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        loss_value = losses.item()

        if not torch.isfinite(losses):
            print(f"[Epoch {epoch}] Step {step} | Non-finite loss: {loss_value}")
            continue

        if USE_AMP:
            scaler.scale(losses).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            losses.backward()
            optimizer.step()

        running_loss += loss_value
        n_steps += 1

        if step % print_freq == 0:
            print(f"[Epoch {epoch}] Step {step}/{len(data_loader)} | Loss: {loss_value:.4f}")

    return running_loss / max(n_steps, 1)

@torch.no_grad()
def evaluate_loss(model, data_loader, device):
    # Trick used in the provided notebooks:
    # keep model.train() so that Faster R-CNN returns the losses.
    model.train()
    total_loss = 0.0
    n_steps = 0

    for images, targets in data_loader:
        images, targets = move_batch_to_device(images, targets, device)

        if USE_AMP:
            with torch.amp.autocast(device_type="cuda"):
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
        else:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        total_loss += losses.item()
        n_steps += 1

    return total_loss / max(n_steps, 1)

def save_checkpoint(path, model, optimizer, scheduler, epoch, train_loss, val_loss,
                    num_classes, dataset, optimizer_name="SGD"):
    checkpoint = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "num_classes": int(num_classes),
        "optimizer_name": optimizer_name,
        "cat_id_to_label": dataset.cat_id_to_label,
        "label_to_cat_id": dataset.label_to_cat_id,
        "label_to_name": dataset.label_to_name,
    }
    torch.save(checkpoint, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None, device="cpu",
                    expected_optimizer_name=None):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    ckpt_optimizer_name = checkpoint.get("optimizer_name")
    same_optimizer = (
        expected_optimizer_name is None
        or ckpt_optimizer_name is None
        or ckpt_optimizer_name == expected_optimizer_name
    )

    if optimizer is not None and same_optimizer and checkpoint.get("optimizer_state_dict") is not None:
        try:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        except Exception as e:
            print("Optimizer loading skipped:", e)

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        try:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        except Exception as e:
            print("Scheduler loading skipped:", e)

    start_epoch = int(checkpoint["epoch"]) + 1
    return checkpoint, start_epoch, checkpoint.get("train_loss"), checkpoint.get("val_loss")

def plot_history(history, title="Training curves"):
    hist_df = pd.DataFrame(history)
    display(hist_df)

    plt.figure(figsize=(8, 5))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train_loss")
    plt.plot(hist_df["epoch"], hist_df["val_loss"], marker="o", label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## 9. Phase 1 — Baseline

Objective: establish a simple reference with Faster R-CNN without specific augmentation.


In [ ]:
# =========================
# PHASE 1 — Baseline
# =========================
model = get_faster_rcnn_model(num_classes).to(device)

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=STEP_SIZE,
    gamma=GAMMA,
)

scaler = torch.amp.GradScaler("cuda") if USE_AMP else None

history_baseline = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "lr": [],
}

best_val_loss = float("inf")
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, optimizer, baseline_train_loader, device, epoch, scaler=scaler, print_freq=20)
    val_loss = evaluate_loss(model, baseline_val_loader, device)
    current_lr = optimizer.param_groups[0]["lr"]
    lr_scheduler.step()

    history_baseline["epoch"].append(epoch)
    history_baseline["train_loss"].append(train_loss)
    history_baseline["val_loss"].append(val_loss)
    history_baseline["lr"].append(current_lr)

    print(f"Epoch {epoch}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | lr={current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        print(f"New best baseline model at epoch {epoch} (val_loss={val_loss:.4f})")

plot_history(history_baseline, title="Baseline training curves")

### Observed history of the Faster R-CNN baseline

|   epoch |   train_loss |   val_loss |     lr |
|--------:|-------------:|-----------:|-------:|
|       1 |       0.2333 |     0.1584 | 0.0050 |
|       2 |       0.1342 |     0.1278 | 0.0050 |
|       3 |       0.1004 |     0.1183 | 0.0050 |
|       4 |       0.0817 |     0.1044 | 0.0050 |
|       5 |       0.0696 |     0.1022 | 0.0050 |
|       6 |       0.0487 |     0.0897 | 0.0005 |
|       7 |       0.0442 |     0.0888 | 0.0005 |
|       8 |       0.0421 |     0.0889 | 0.0005 |
|       9 |       0.0402 |     0.0893 | 0.0005 |
|      10 |       0.0386 |     0.0896 | 0.0005 |


### Baseline interpretation

- The training loss decreases steadily.
- `val_loss` improves mainly up to around epochs 6–7.
- From that point onward, the gain becomes marginal: this is a **plateau signal**.
- This reading justifies the decision not to keep extending the same pipeline indefinitely without a structural change.


## 10. Phase 2 — Data augmentation

Objective: improve model generalization by enriching the variability of the images seen during training.

In this consolidated version, I propose a reasonable and easy-to-maintain augmentation:
- random horizontal flip;
- slight brightness variation;
- slight contrast variation.

These choices remain compatible with an object detection problem, as long as the boxes are correctly updated.


In [ ]:
# =========================
# PHASE 2 — Data augmentation
# =========================
model_aug = get_faster_rcnn_model(num_classes).to(device)

optimizer_aug = torch.optim.SGD(
    [p for p in model_aug.parameters() if p.requires_grad],
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

lr_scheduler_aug = torch.optim.lr_scheduler.StepLR(
    optimizer_aug,
    step_size=STEP_SIZE,
    gamma=GAMMA,
)

scaler_aug = torch.amp.GradScaler("cuda") if USE_AMP else None

history_aug = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "lr": [],
}

best_val_loss_aug = float("inf")
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model_aug, optimizer_aug, aug_train_loader, device, epoch, scaler=scaler_aug, print_freq=20)
    val_loss = evaluate_loss(model_aug, aug_val_loader, device)
    current_lr = optimizer_aug.param_groups[0]["lr"]
    lr_scheduler_aug.step()

    history_aug["epoch"].append(epoch)
    history_aug["train_loss"].append(train_loss)
    history_aug["val_loss"].append(val_loss)
    history_aug["lr"].append(current_lr)

    print(f"Epoch {epoch}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | lr={current_lr:.6f}")

    if val_loss < best_val_loss_aug:
        best_val_loss_aug = val_loss
        print(f"New best augmentation model at epoch {epoch} (val_loss={val_loss:.4f})")

plot_history(history_aug, title="Data augmentation curves")

### Observed history in the provided data augmentation lab

|   epoch |   train_loss |   val_loss |     lr |
|--------:|-------------:|-----------:|-------:|
|       1 |       0.1425 |     0.1250 | 0.0050 |
|       2 |       0.1120 |     0.1235 | 0.0050 |
|       3 |       0.0939 |     0.1020 | 0.0050 |
|       4 |       0.0813 |     0.1031 | 0.0050 |
|       5 |       0.0610 |     0.0878 | 0.0005 |
|       6 |       0.0563 |     0.0866 | 0.0005 |
|       7 |       0.0547 |     0.0866 | 0.0005 |
|       8 |       0.0529 |     0.0869 | 0.0005 |
|       9 |       0.0514 |     0.0865 | 0.0005 |
|      10 |       0.0493 |     0.0860 | 0.00005 |


### Interpretation of the data augmentation phase

- The observed minimum `val_loss` drops to **0.0860**.
- This is better than the observed baseline (**0.0888**).
- The improvement is small in absolute terms, but it is consistent with your experimental conclusion: **data augmentation improves generalization**.
- This is therefore the best variant to retain as the main model.


## 11. Phase 3 — Fine-tuning

Objective: start again from the best model with data augmentation and continue with a lower learning rate, a `ReduceLROnPlateau` scheduler, and an early stopping mechanism.


In [ ]:
# =========================
# PHASE 3 — Fine-tuning
# =========================
# Idea: restart from the best checkpoint of the data augmentation phase,
# then continue with a lower LR.
#
# Option A: unfreeze the whole model (as in your trials if all parameters were trainable)
# Option B: unfreeze only some backbone / head layers
#
# Here I propose simple global fine-tuning.

model_ft = get_faster_rcnn_model(num_classes).to(device)

# Example if you saved the best augmentation checkpoint
AUG_BEST_CKPT = Path("/content/checkpoints_local/fasterrcnn_sgd_aug_best.pt")
if AUG_BEST_CKPT.exists():
    checkpoint, _, _, best_val_loss_from_aug = load_checkpoint(
        AUG_BEST_CKPT,
        model_ft,
        device=device,
        expected_optimizer_name="SGD",
    )
    print("Augmentation checkpoint loaded. Initial best_val_loss =", best_val_loss_from_aug)
else:
    checkpoint = {}
    best_val_loss_from_aug = float("inf")
    print("Augmentation checkpoint not found: fine-tuning will restart from the ImageNet model.")

optimizer_ft = torch.optim.SGD(
    [p for p in model_ft.parameters() if p.requires_grad],
    lr=FINE_TUNE_LR,
    momentum=FINE_TUNE_MOMENTUM,
    weight_decay=FINE_TUNE_WEIGHT_DECAY,
)

lr_scheduler_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ft,
    mode="min",
    factor=PLATEAU_FACTOR,
    patience=PLATEAU_PATIENCE,
    threshold=MIN_DELTA,
    threshold_mode="abs",
    min_lr=1e-5,
)

scaler_ft = torch.amp.GradScaler("cuda") if USE_AMP else None

history_ft = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "lr": [],
}

best_val_loss_ft = float(best_val_loss_from_aug)
epochs_without_improvement = 0

for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    train_loss = train_one_epoch(model_ft, optimizer_ft, aug_train_loader, device, epoch, scaler=scaler_ft, print_freq=20)
    val_loss = evaluate_loss(model_ft, aug_val_loader, device)
    current_lr = optimizer_ft.param_groups[0]["lr"]
    lr_scheduler_ft.step(val_loss)

    history_ft["epoch"].append(epoch)
    history_ft["train_loss"].append(train_loss)
    history_ft["val_loss"].append(val_loss)
    history_ft["lr"].append(current_lr)

    print(f"[Fine-tuning] Epoch {epoch}/{FINE_TUNE_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | lr={current_lr:.6f}")

    if val_loss < (best_val_loss_ft - MIN_DELTA):
        best_val_loss_ft = val_loss
        epochs_without_improvement = 0
        print(f"New best fine-tuning model at epoch {epoch} (val_loss={val_loss:.4f})")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping: insufficient validation improvement.")
        break

plot_history(history_ft, title="Fine-tuning curves")

### Observed history in the provided fine-tuning lab

|   epoch |   train_loss |   val_loss |    lr |
|--------:|-------------:|-----------:|------:|
|       1 |       0.0517 |     0.0873 | 0.001 |
|       2 |       0.0498 |     0.0873 | 0.001 |
|       3 |       0.0480 |     0.0864 | 0.001 |


### Fine-tuning interpretation

- Fine-tuning continues to reduce `train_loss`.
- `val_loss` stays close to the best level reached with augmentation, but **does not clearly surpass it** in the available outputs.
- The best `val_loss` observed in the visible fine-tuning outputs is **0.0864**, compared with **0.0860** for data augmentation.
- The practical conclusion therefore remains: **keep the data augmentation model as the main reference**.


## 12. Reproducible comparison from the observed values

This cell reconstructs the comparison directly from the logs visible in the three labs.


In [ ]:
# =========================
# Reproducible comparison based on the results observed in the 3 labs
# =========================
baseline_observed = [
  {
    "epoch": 1,
    "train_loss": 0.2333,
    "val_loss": 0.1584,
    "lr": 0.005
  },
  {
    "epoch": 2,
    "train_loss": 0.1342,
    "val_loss": 0.1278,
    "lr": 0.005
  },
  {
    "epoch": 3,
    "train_loss": 0.1004,
    "val_loss": 0.1183,
    "lr": 0.005
  },
  {
    "epoch": 4,
    "train_loss": 0.0817,
    "val_loss": 0.1044,
    "lr": 0.005
  },
  {
    "epoch": 5,
    "train_loss": 0.0696,
    "val_loss": 0.1022,
    "lr": 0.005
  },
  {
    "epoch": 6,
    "train_loss": 0.0487,
    "val_loss": 0.0897,
    "lr": 0.0005
  },
  {
    "epoch": 7,
    "train_loss": 0.0442,
    "val_loss": 0.0888,
    "lr": 0.0005
  },
  {
    "epoch": 8,
    "train_loss": 0.0421,
    "val_loss": 0.0889,
    "lr": 0.0005
  },
  {
    "epoch": 9,
    "train_loss": 0.0402,
    "val_loss": 0.0893,
    "lr": 0.0005
  },
  {
    "epoch": 10,
    "train_loss": 0.0386,
    "val_loss": 0.0896,
    "lr": 0.0005
  }
]
augmentation_observed = [
  {
    "epoch": 1,
    "train_loss": 0.1425,
    "val_loss": 0.125,
    "lr": 0.005
  },
  {
    "epoch": 2,
    "train_loss": 0.112,
    "val_loss": 0.1235,
    "lr": 0.005
  },
  {
    "epoch": 3,
    "train_loss": 0.0939,
    "val_loss": 0.102,
    "lr": 0.005
  },
  {
    "epoch": 4,
    "train_loss": 0.0813,
    "val_loss": 0.1031,
    "lr": 0.005
  },
  {
    "epoch": 5,
    "train_loss": 0.061,
    "val_loss": 0.0878,
    "lr": 0.0005
  },
  {
    "epoch": 6,
    "train_loss": 0.0563,
    "val_loss": 0.0866,
    "lr": 0.0005
  },
  {
    "epoch": 7,
    "train_loss": 0.0547,
    "val_loss": 0.0866,
    "lr": 0.0005
  },
  {
    "epoch": 8,
    "train_loss": 0.0529,
    "val_loss": 0.0869,
    "lr": 0.0005
  },
  {
    "epoch": 9,
    "train_loss": 0.0514,
    "val_loss": 0.0865,
    "lr": 0.0005
  },
  {
    "epoch": 10,
    "train_loss": 0.0493,
    "val_loss": 0.086,
    "lr": 5e-05
  }
]
finetuning_observed = [
  {
    "epoch": 1,
    "train_loss": 0.0517,
    "val_loss": 0.0873,
    "lr": 0.001
  },
  {
    "epoch": 2,
    "train_loss": 0.0498,
    "val_loss": 0.0873,
    "lr": 0.001
  },
  {
    "epoch": 3,
    "train_loss": 0.048,
    "val_loss": 0.0864,
    "lr": 0.001
  }
]

df_base = pd.DataFrame(baseline_observed)
df_aug = pd.DataFrame(augmentation_observed)
df_ft = pd.DataFrame(finetuning_observed)

summary = pd.DataFrame([
    {
        "approach": "Baseline",
        "observed_epochs": len(df_base),
        "best_epoch": int(df_base.loc[df_base.val_loss.idxmin(), "epoch"]),
        "best_val_loss": float(df_base.val_loss.min()),
    },
    {
        "approach": "Data augmentation",
        "observed_epochs": len(df_aug),
        "best_epoch": int(df_aug.loc[df_aug.val_loss.idxmin(), "epoch"]),
        "best_val_loss": float(df_aug.val_loss.min()),
    },
    {
        "approach": "Fine-tuning",
        "observed_epochs": len(df_ft),
        "best_epoch": int(df_ft.loc[df_ft.val_loss.idxmin(), "epoch"]),
        "best_val_loss": float(df_ft.val_loss.min()),
    },
])

display(summary)

plt.figure(figsize=(9, 5))
plt.plot(df_base["epoch"], df_base["val_loss"], marker="o", label="Baseline")
plt.plot(df_aug["epoch"], df_aug["val_loss"], marker="o", label="Data augmentation")
plt.plot(df_ft["epoch"], df_ft["val_loss"], marker="o", label="Fine-tuning")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Comparison of validation losses")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 13. Prediction visualization

This function generates qualitative visualizations on a few validation images to compare multiple checkpoints.


In [ ]:
def show_predictions(model, data_loader, device, label_to_name, score_threshold=0.5, max_images=4):
    model.eval()
    images, targets = next(iter(data_loader))
    images_device = [img.to(device) for img in images[:max_images]]

    with torch.no_grad():
        outputs = model(images_device)

    for img_tensor, output in zip(images[:max_images], outputs):
        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
        fig, ax = plt.subplots(1, figsize=(10, 8))
        ax.imshow(img_np)

        boxes = output["boxes"].detach().cpu()
        labels = output["labels"].detach().cpu()
        scores = output["scores"].detach().cpu()

        kept = 0
        for box, label, score in zip(boxes, labels, scores):
            if score.item() < score_threshold:
                continue

            x1, y1, x2, y2 = box.tolist()
            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor="red",
                facecolor="none",
            )
            ax.add_patch(rect)

            class_name = label_to_name.get(int(label), f"class_{int(label)}")
            ax.text(
                x1,
                max(0, y1 - 5),
                f"{class_name} {score.item():.2f}",
                color="white",
                fontsize=10,
                bbox=dict(facecolor="red", alpha=0.6, edgecolor="none"),
            )
            kept += 1

        ax.set_title(f"Kept predictions: {kept}")
        ax.axis("off")
        plt.show()

## Analytical conclusion

Based on the **results observed in the provided notebooks**, the most robust reading is the following:

1. The **baseline Faster R-CNN** learns correctly, with a clear drop in `train_loss` and an initial improvement in `val_loss`.
2. Adding extra epochs after the first plateau does **not seem to significantly change generalization**.
3. The **data augmentation** approach provides the best measurable improvement among the available histories, with an observed `best_val_loss` of **0.0860** versus **0.0888** for the baseline.
4. **Fine-tuning** provides at best a minor local improvement in the visible outputs, but **remains below the best run with data augmentation**.
